# 01 — Data Exploration
**Financial Risk Model | Exploratory Data Analysis**

This notebook explores the raw financial dataset to understand:
- Dataset structure, size, and schema
- Missing values and data quality issues
- Feature distributions and outliers
- Class balance (Low Risk vs High Risk)
- Bivariate relationships between features and the target
- Correlation structure among numeric features

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
sys.path.insert(0, os.path.join('..', 'utils'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import generate_synthetic_dataset, load_raw_data
from helpers import describe_dataframe, profile_missing, value_counts_pct

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 50)

print('Libraries loaded ✅')

## 1. Load Dataset

In [ ]:
# Generate synthetic data if raw file doesn't exist yet
RAW_PATH = '../data/raw/financial_risk_data.csv'

if not os.path.exists(RAW_PATH):
    print('Generating synthetic dataset ...')
    df = generate_synthetic_dataset(n_samples=5000, save_path=RAW_PATH)
else:
    df = load_raw_data(filepath=RAW_PATH, config_path='../config/config.yaml')

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 2. Dataset Overview

In [ ]:
describe_dataframe(df, target_col='RiskFlag')

In [ ]:
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# Missing value profile
missing = profile_missing(df)
if missing.empty:
    print('✅ No missing values in the dataset.')
else:
    display(missing)

## 3. Target Variable — Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
counts = df['RiskFlag'].value_counts()
labels = ['Low Risk (0)', 'High Risk (1)']
colors = ['#4CAF50', '#F44336']
axes[0].bar(labels, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=11)
axes[0].set_title('Class Distribution — RiskFlag', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Class Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

imbalance_ratio = counts[0] / counts[1]
print(f'Class imbalance ratio: {imbalance_ratio:.2f}:1 (non-default:default)')
print('\n⚠️  Consider class-weight balancing or SMOTE oversampling during training.')

## 4. Numeric Feature Distributions

In [ ]:
numeric_cols = ['Age', 'Income', 'LoanAmount', 'LoanTenure', 'CreditScore',
                'ExistingLoansCount', 'MonthlyEMI', 'TotalAssets', 'TotalLiabilities', 'NetMonthlyIncome']

fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    # KDE by risk class
    for flag, color, label in [(0, '#4CAF50', 'Low Risk'), (1, '#F44336', 'High Risk')]:
        subset = df[df['RiskFlag'] == flag][col].dropna()
        subset.plot.kde(ax=ax, color=color, label=label, linewidth=2)
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=9)

# Hide unused subplots
for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature Distributions by Risk Class', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Categorical Feature Analysis

In [ ]:
cat_cols = ['EmploymentType', 'MaritalStatus', 'EducationLevel', 'PropertyOwnership', 'LoanPurpose']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    ax = axes[i]
    risk_rate = df.groupby(col)['RiskFlag'].mean().sort_values(ascending=False)
    bars = ax.bar(risk_rate.index, risk_rate.values * 100,
                  color=plt.cm.RdYlGn_r(risk_rate.values), edgecolor='white')
    ax.set_title(f'Default Rate by {col}', fontsize=11, fontweight='bold')
    ax.set_ylabel('Default Rate (%)')
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, risk_rate.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val*100:.1f}%', ha='center', fontsize=9)

axes[-1].set_visible(False)
plt.suptitle('Default Rate by Categorical Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/categorical_default_rates.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Correlation Heatmap

In [ ]:
numeric_df = df[numeric_cols + ['RiskFlag']].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(numeric_df, dtype=bool))
sns.heatmap(
    numeric_df, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
    annot_kws={'size': 9}
)
ax.set_title('Correlation Matrix (including RiskFlag target)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Top correlations with target
print('\nTop 10 correlations with RiskFlag:')
target_corr = numeric_df['RiskFlag'].drop('RiskFlag').sort_values(key=abs, ascending=False)
print(target_corr.head(10).to_string())

## 7. Outlier Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

high_variance_cols = ['Income', 'LoanAmount', 'MonthlyEMI', 'TotalAssets',
                      'TotalLiabilities', 'CreditScore', 'Age', 'LoanTenure']

for i, col in enumerate(high_variance_cols):
    ax = axes[i]
    df.boxplot(column=col, by='RiskFlag', ax=ax,
               boxprops=dict(color='#2196F3'),
               medianprops=dict(color='#FF5722', linewidth=2),
               whiskerprops=dict(color='#2196F3'),
               capprops=dict(color='#2196F3'))
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_xlabel('RiskFlag (0=Low, 1=High)')

plt.suptitle('Box Plots — Numeric Features by Risk Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Key Findings Summary

In [ ]:
print('=' * 60)
print('  EDA KEY FINDINGS')
print('=' * 60)
print(f'Dataset size     : {len(df):,} records')
print(f'Features         : {df.shape[1] - 2} (excl. CustomerID and target)')
print(f'Default rate     : {df["RiskFlag"].mean()*100:.1f}%')
print(f'Missing values   : {df.isnull().sum().sum()} total')

top_corr_feat = target_corr.index[0]
top_corr_val  = target_corr.iloc[0]
print(f'Strongest signal : {top_corr_feat} (r={top_corr_val:.3f})')
print()
print('Next steps → Feature Engineering (notebook 02)')